# Predicting Request Nature and Resolution Time for Montreal 311 Service Requests

## Abstract



This project studies whether Montreal 311 service requests can be modeled in a way that is actually useful under a realistic time-based evaluation setup. I focused on two tasks: predicting the request nature (`NATURE`) from information available at creation time, and predicting time-to-resolution in days for requests that eventually close. The project uses the Montreal open-data 311 dataset and compares simple baseline models.

For classification, the strongest rerun model was a calibrated linear SVM on the combined sparse feature view.

For regression, the best rerun model was a conditional-median lookup baseline. It performed slightly better than the linear regression baselines on the validation split.

The final results show very strong performance on the dominant request classes, but much weaker results on the minority classes. In the rerun outputs, the classifier reached about 0.505 macro-F1 on the test split while the regression baseline reached about 42.3 days MAE.

For regression, the conformal interval reached essentially the nominal 90% coverage, but the intervals were still wide. Overall, my project supports the idea that creation-time metadata can be useful for both routing and rough resolution-time estimation, but the results also show clear limits in class imbalance and uncertainty precision.


## Introduction


The main motivation for this project comes from a practical municipal problem I personally have experienced while dealing with the municipality of my borrough: if a request is classified incorrectly, it can be routed through the wrong process, and if the expected resolution time is unrealistic, that can waste resources and frustrate citizens. In the proposal, I framed the project around those two issues and asked whether request information available at creation time was enough to support useful prediction.



The project uses the Montreal 311 open-data dataset covering requests from 2019 to 2021. The two main prediction tasks stayed the same from the proposal to the final project: predict the request nature, and predict the resolution time for closed-like requests. I also kept the same general evaluation idea from the proposal: a chronological split instead of a random split, so that training is done on older requests and testing is done on newer ones.



In the proposal, I noted that a lot of 311-related work and reporting examples focus more on descriptive analysis and visualization than on a careful predictive setup. Because of that, I wanted this project to emphasize simple predictive models, leakage control, calibration, and realistic evaluation across time. The final project is close to that goal, even though i had to simplify (or drop) some planned extensions.



The final results show that the problem is partly learnable, but not equally across all targets. Classification works well for the dominant classes, while minority classes remain difficult. Regression produces useful rough estimates and well-calibrated coverage, but not narrow intervals.


## Methodology


The project has two supervised-learning pipelines with shared preparation steps and a shared chronological split. I first cleaned the raw CSV, normalized the target labels, parsed the timestamps, and built a representative subset so that the experiments were easier to rerun. I then prepared separate views for classification and regression, because the regression task only makes sense for requests that reached a closed-like status.



For classification, I used simple linear baselines and compared different feature views. The final workflow includes text-only, tabular-only, and combined sparse feature views, plus a calibrated linear SVM baseline. 

For regression, I used simple baselines as well: a dummy mean baseline, a conditional-median lookup baseline, and linear baselines based on ridge and elastic net with a training-only log transform of the target.



A major part of the methodology was trying to keep the evaluation realistic. I evaluated on the final test split once. I also added reporting tools that made the final results easier to interpret: class-level summaries, confusion matrices, grouped slice summaries, coefficient-based feature summaries, and simple uncertainty outputs.



### Consistency with the proposal



A lot of the final project stayed close to the original plan:



- the dataset and the two main prediction tasks stayed the same;

- the evaluation stayed chronological and not random;

- linear models styed the main baseline family;

- calibration and uncertainty stayed part of the final project.



### Main deviations from the proposal



There are also a few clear differences between the proposal and the finished project:



- in the proposal I mentioned a subset closer to 250K rows, while the final workflow is a representative 300K subset.


- I mentioned trying TabNet as a stronger deep tabular baseline. That was not included in the final project. I stayed focused on simpler baselines in my project.


- I wanted to originally do a more explicit high-risk case review based on flagged examples. In the final project, i handled that idea through reliability tables, class-level summaries, slice summaries, and the written discussion.


- The proposal described uncertainty in a broader way. In the final version, this morphed into a classification reliability summary and regression conformal prediction intervals.



So the final project is still very close to the proposal in spirit, but it ended up being more focused and more baseline-oriented than originally planned.


## Experimental Setup


The main dataset is the Montreal 311 open-data file `requetes311_2019-2021.csv`. To keep the experiments manageable and reproducible, I worked from a representative 300K subset built from the raw file. The classification task uses rows with a valid creation timestamp and a normalized `NATURE` label. The regression task uses only closed-like requests with valid creation and final-status timestamps, and the target is time-to-resolution in days.



The split is chronological and not random. Training uses requests up to the end of 2020, validation uses the first half of 2021, and testing uses the second half of 2021. This is important because the project is meant to simulate future prediction from historical data.



The main classification baselines are a dummy prior baseline, logistic regression on text-only features, logistic regression on tabular-only features, logistic regression on the combined sparse feature view, and a calibrated linear SVM on the combined sparse feature view.



The main regression baselines are a dummy mean baseline, a conditional median lookup baseline, ridge regression with a training-only log target transform, and elastic net regression with the same stabilized target approach.



Some key settings from the final code are: logistic regression uses the `saga` solver with class balancing; the calibrated SVM uses `LinearSVC` plus 3-fold calibration; elastic net uses a small regularization setting with a fixed random state; classification is judged mainly with macro-F1 plus class-level summaries and a multiclass Brier score; regression is judged mainly with MAE and RMSE, plus a 90% conformal interval summary.


## Notebook Setup and Reproducibility


This notebook is the main submission file. The repository should include this notebook, the root `requirements.txt` file, and the saved output files that feed the tables and figures below.



The purpose of this notebook is to reproduce the reported results from committed saved outputs instead of retraining everything from scratch.

In [ ]:
# Choose the notebook mode and set up the repo location.
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/WaRRatic/COMP-432.git"
REPO_DIR_NAME = "montreal311_project_repo"

if "google.colab" in sys.modules:
    print("proceeding in Google Colab mode")
    repo_root = Path.cwd() / REPO_DIR_NAME
    subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(repo_root / "requirements.txt")],
        check=True,
    )
else:
    print("proceeding in local mode")
    repo_root = Path.cwd().parent

repo_root


In [ ]:
# Point the notebook at the project folders that hold the saved outputs.
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = repo_root / "Project"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
CLASSIFICATION_DIR = OUTPUTS_DIR / "classification"
REGRESSION_DIR = OUTPUTS_DIR / "regression"
ANALYSIS_DIR = OUTPUTS_DIR / "analysis"
PROJECT_ROOT


In [ ]:
# Load the saved metrics and tables that feed the rest of the report.
def load_json(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

classification_summary = load_json(CLASSIFICATION_DIR / 'summary.json')
regression_summary = load_json(REGRESSION_DIR / 'summary.json')

classification_models = pd.read_csv(CLASSIFICATION_DIR / 'model_comparison.csv')
classification_report = pd.read_csv(CLASSIFICATION_DIR / 'best_model_class_report.csv')
classification_reliability = pd.read_csv(CLASSIFICATION_DIR / 'best_model_reliability.csv')
classification_by_month = pd.read_csv(ANALYSIS_DIR / 'classification_by_month.csv')
classification_by_class = pd.read_csv(ANALYSIS_DIR / 'classification_by_class.csv')

regression_models = pd.read_csv(REGRESSION_DIR / 'model_comparison.csv')
conformal_predictions = pd.read_csv(REGRESSION_DIR / 'conformal_test_predictions.csv')

## Experimental Results



The tables below summarize the overall model comparison first. I use them to identify the strongest saved classification and regression models anf then mobe into the more detailed class-level and slice-level breakdowns.



One important point is that the classification accuracy is high, but the dataset is heavily imbalanced, so macro-F1 and class-level results give a much more honest picture of performance.


In [ ]:
# Pull the top-line classification and regression results into compact summary tables.
overview = pd.DataFrame([
    {
        'task': 'classification',
        'best_model': classification_summary['best_model'],
        'main_metric_1': classification_summary['final_test_metrics']['accuracy'],
        'main_metric_2': classification_summary['final_test_metrics']['macro_f1'],
        'notes': 'accuracy and macro F1'
    },
    {
        'task': 'regression',
        'best_model': regression_summary['best_model'],
        'main_metric_1': regression_summary['final_test_metrics']['mae'],
        'main_metric_2': regression_summary['final_test_metrics']['rmse'],
        'notes': 'MAE and RMSE in days'
    }
])

display(Markdown('### Final test summary'))
display(overview)

display(Markdown('### Validation comparison for classification'))
display(classification_models.sort_values('validation_macro_f1', ascending=False))

display(Markdown('### Validation comparison for regression'))
display(regression_models.sort_values('validation_mae'))

interval_summary = pd.DataFrame([
    {
        'alpha': regression_summary['conformal_interval']['alpha'],
        'nominal_coverage': regression_summary['conformal_interval']['nominal_coverage'],
        'observed_coverage': regression_summary['conformal_interval']['test_metrics']['coverage'],
        'interval_radius_days': regression_summary['conformal_interval']['interval_radius_days'],
        'mean_interval_width_days': regression_summary['conformal_interval']['test_metrics']['mean_interval_width']
    }
])

display(Markdown('### Regression interval summary'))
display(interval_summary)

### Class-Level Behavior


After the overall comparison, I looked at the results in more detail. The next outputs show class-level classification behavior, reliability, minority-class risk, and example regression intervals. This matters because the project is not equally strong across all parts of the dataset.


In [ ]:
# Surface the class-level details and a small sample of regression intervals.
display(Markdown('### Classification report by class'))
display(classification_report)

display(Markdown('### Reliability table for the best classifier'))
display(classification_reliability)

minority_classes = classification_by_class.sort_values('support')
display(Markdown('### Minority-class classification risk'))
display(minority_classes[minority_classes['support'] < 1000])

display(Markdown('### Example regression prediction intervals'))
display(conformal_predictions.head(10))

In [ ]:
# Turn the saved results into a few quick visual summaries.
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

classification_report.set_index('label')['f1'].plot(
    kind='bar',
    ax=axes[0],
    color='#2a6f97',
    title='Classification F1 by class'
)
axes[0].set_ylabel('F1')
axes[0].tick_params(axis='x', rotation=45)

classification_by_month_plot = classification_by_month.rename(columns={'group': 'month'})
classification_by_month_plot.plot(
    x='month',
    y='macro_f1',
    marker='o',
    ax=axes[1],
    color='#ef8354',
    title='Classification macro F1 by month'
)
axes[1].set_ylabel('Macro F1')
axes[1].tick_params(axis='x', rotation=45)

interval_sample = conformal_predictions.head(60).copy()
interval_sample = interval_sample.sort_values('prediction').reset_index(drop=True)
interval_x = range(len(interval_sample))
axes[2].fill_between(
    interval_x,
    interval_sample['lower_bound'],
    interval_sample['upper_bound'],
    color='#4f772d',
    alpha=0.25,
    label='prediction interval'
)
axes[2].plot(
    interval_x,
    interval_sample['prediction'],
    color='#2f3e46',
    linewidth=1.5,
    label='prediction'
)
axes[2].scatter(
    interval_x,
    interval_sample['actual'],
    color='#bc4749',
    s=12,
    alpha=0.8,
    label='actual'
)
axes[2].set_title('Regression intervals sample')
axes[2].set_xlabel('Sampled test rows')
axes[2].set_ylabel('Days')
axes[2].legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## Conclusions



Overall, I think the project reached the main goals from the proposal in a realistic way. I built a reproducible subset, used a chronological evaluation setup, compared multiple baseline models for both tasks, and added uncertainty-related reporting instead of relying only on one final score.



For classification, the strongest final result came from the calibrated linear SVM, and the final accuracy was very high. But the class-level analysis makes it clear that this does not mean the problem is solved equally well for every request type. `Information` and `Requete` dominate the data and are handled well, while `Plainte` and `Commentaire` remain much harder.



For regression, the final result is a bit more surprising to me: the conditional-median lookup baseline ended up outperforming the linear regression baselines on the validation split. That suggests that for this target, simple grouped historical information is already very strong. The conformal interval also achieved close to the intended 90% coverage as discussed in my proposal, but the interval width shows that the uncertainty is still broad.



The main things I could not fully complete from the proposal were the TabNet comparison and the more detailed high-risk case review based on flagged examples. So the finished project is a bit narrower than originally planned. Still, I think it answers the main project question well enough: creation-time metadata can support useful baseline prediction for Montreal 311 requests, but the final outputs need to be interpreted carefully, especially under class imbalance and wide uncertainty ranges.


## References


My proposal already includes a longer reference list, and I reused some of those sources here. These are the references most directly tied to the dataset, the main modeling choices, the uncertainty method, and the implementation work in this project.



1. Ville de Montréal, `Demandes de services citoyennes (Requêtes 311)`, Données Montréal Open Data Portal.
    - https://donnees.montreal.ca/dataset/requete-311

2. scikit-learn documentation for `LogisticRegression`, `LinearSVC`, `CalibratedClassifierCV`, preprocessing utilities, and evaluation metrics and other topics:
    - https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
    - https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
    - https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html
    - https://scikit-learn.org/stable/modules/preprocessing.html
    - https://scikit-learn.org/stable/modules/model_evaluation.html
    - https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html
    - https://scikit-learn.org/stable/common_pitfalls.html#data-leakage
    - https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html
    - https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html
    - https://scikit-learn.org/stable/modules/model_evaluation.html#brier-score-loss
    - https://scikit-learn.org/stable/modules/generated/sklearn.compose.TransformedTargetRegressor.html
    - https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html
    - https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html

3. pandas documentation on the following topics:
    - https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
    - https://pandas.pydata.org/docs/user_guide/scale.html
    - https://pandas.pydata.org/docs/user_guide/io.html

4. python documentation:
    - https://docs.python.org/3/library/pathlib.html
    - https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/

5. NumPy documentation for `log1p`, used in the stabilized regression target.
    - https://numpy.org/doc/stable/reference/generated/numpy.log1p.html

6. G. Shafer and V. Vovk, `A tutorial on conformal prediction`, Journal of Machine Learning Research, 2008.
    - https://jmlr.csail.mit.edu/papers/v9/shafer08a.html

7. N. Kruchten, `Montreal 311`, blog post used as a related descriptive example.
    - https://nicolas.kruchten.com/content/2015/06/montreal-311/

8. COMP 432 course materials (labs, lectures, tutorials, and additional material provided by the professor).


### Coding Source Acknowledgment


Most of the final code in this project was written specifically for the course project. The outside sources I relied on were mainly documentation and reference material rather than full code to copy from.



The main coding references were:



- scikit-learn documentation, especially for preprocessing pipelines, linear models, calibration, and metrics;

- pandas documentation for loading CSV files, datetime handling, grouping, and summary tables;

- matplotlib documentation for the report plots;

- the course labs and notebook style used in COMP 432;

- the proposal references already listed above, especially the conformal prediction tutorial and Montreal 311 dataset pages.



I did not copy large external code blocks directly into the project. When outside material was used, it was mainly to check API usage, implementation details, and expected behavior.
